# LoRA Finetuning of `DenoiserWrapper` (toy example)

Minimal end-to-end example of finetuning the dynamics denoiser with [PEFT](https://github.com/huggingface/peft) LoRA adapters.

Steps:
1. Load the pretrained `DenoiserWrapper` (same recipe as `sampling.ipynb`).
2. Inspect attention/FFN module names so we know what to target.
3. Wrap the denoiser with `peft.get_peft_model` — base weights freeze, LoRA matrices become the only trainable params.
4. Run a few optimizer steps on real data using the existing `UWMForwardProcess` + `compute_uwm_loss`.
5. Save / reload the adapter and merge it into the base for inference.

Install once:
```bash
pip install peft
```

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Load pretrained denoiser + tokenizer + dataset

In [2]:
import torch
import torch.nn as nn
from hydra import initialize, compose
from omegaconf import OmegaConf

from dreamerv4uwm.datasets import ShardedHDF5Dataset
from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser
from dreamerv4uwm.loss import UWMForwardProcess, compute_uwm_loss

DATA_PATH = "/media/mim-server/Volume1/lewm/sharded/cube/"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
with initialize(version_base=None, config_path="../scripts/config"):
    cfg = compose(config_name="dynamics/lewm-cubes.yaml")

cfg.denoiser.layer_types = ["spatial", "temporal", "spatial", "temporal"]
cfg.dynamics_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/dynamics/lewm-cubes/joint/250000.pt"
cfg.tokenizer_ckpt = "/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer_ckpts/soar.pt"

denoiser = load_denoiser(cfg, device)
tokenizer = load_tokenizer(cfg, device).eval()
denoiser.train();  # enable grads on attention / FFN; LoRA params will be the only trainable ones

/home/mim-server/miniconda3/envs/dreamerv4/lib/python3.11/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'dynamics/lewm-cubes.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [4]:
from torch.utils.data import DataLoader
from torch.nn.functional import interpolate

WINDOW = 16   # short window for a toy run
BATCH  = 2
RES    = (256, 256)

dataset = ShardedHDF5Dataset(
    data_dir=DATA_PATH,
    window_size=WINDOW,
    stride=1,
    split='train',
    train_fraction=0.9,
    split_seed=123,
)
loader = DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
len(dataset)

Train split: 1665000 windows from 9000 episodes


1665000

## 2. Inspect the module tree
We want to know the **suffix names** of the linear layers we will inject LoRA into. Standard practice for transformers: the four attention projections `W_q`, `W_k`, `W_v`, `W_o`. Optionally also the FFN `up`, `gate`, `down`.

In [5]:
# Print the unique linear-layer suffix names so we know what to target
from collections import Counter
linear_suffixes = Counter()
for name, mod in denoiser.named_modules():
    if isinstance(mod, nn.Linear):
        linear_suffixes[name.rsplit('.', 1)[-1]] += 1
linear_suffixes

Counter({'W_q': 24,
         'W_k': 24,
         'W_v': 24,
         'W_o': 24,
         'up': 24,
         'gate': 24,
         'down': 24,
         'latent_projector': 1,
         'obs_projector': 1,
         'action_projector': 1,
         'obs_diff_control_proj': 1,
         'act_diff_control_proj': 1,
         'action_input_proj': 1})

## 3. Wrap with PEFT LoRA
`get_peft_model` walks `named_modules()` and replaces every `nn.Linear` whose name matches a `target_modules` suffix with a `LoraLinear`: it keeps the original frozen `nn.Linear` and adds two small trainable matrices `A: (r, in)` and `B: (out, r)`. The forward becomes `Wx + (alpha/r) * B @ A @ x`.

Three common targeting strategies — pick **one** of the cells below:

- **Conservative (Q/V)** — fewest params, often enough for distribution shift.
- **Attention-only (Q/K/V/O)** — standard transformer LoRA recipe.
- **All linear layers** — `target_modules="all-linear"` makes PEFT wrap every `nn.Linear` it finds: attention (`W_q/W_k/W_v/W_o`), FFN (`up/gate/down`), the head projectors (`obs_projector`, `action_projector`, `latent_projector`, `action_input_proj`), and the control-token combiners (`obs_diff_control_proj`, `act_diff_control_proj`). Maximum capacity, ~1–2% trainable. Note: `DiscreteEmbedder` uses raw `nn.Parameter`, not `nn.Linear`/`nn.Embedding`, so it stays frozen — add it to `modules_to_save` if you want to adapt the noise-level embeddings too.

In [6]:
from peft import LoraConfig, get_peft_model

# --- pick one ---
# (a) conservative
# target = ["W_q", "W_v"]

# (b) attention-only
# target = ["W_q", "W_k", "W_v", "W_o"]

# (c) every nn.Linear in the model (recommended starting point for "LoRA-friendly everywhere")
target = "all-linear"

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=target,
    # modules_to_save=[...]  # e.g. ["obs_projector"] to fully fine-tune (no LoRA) specific modules
)
denoiser = get_peft_model(denoiser, lora_cfg)
denoiser.print_trainable_parameters()

trainable params: 2,863,696 || all params: 175,398,741 || trainable%: 1.6327


In [7]:
# Sanity check: only LoRA params should require grad
n_trainable = sum(p.numel() for p in denoiser.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in denoiser.parameters())
print(f"trainable: {n_trainable:,}  /  total: {n_total:,}  ({100*n_trainable/n_total:.3f}%)")

# A few of the actually-trainable param names:
for n, p in denoiser.named_parameters():
    if p.requires_grad:
        print(n, tuple(p.shape))
        break

trainable: 2,863,696  /  total: 175,398,741  (1.633%)
base_model.model.model.layers.0.layers.0.attn.attn.W_q.lora_A.default.weight (8, 768)


## 4. Toy training loop
Reuses the same forward-process / loss the main trainer uses. Tokenize the batch with the (frozen) tokenizer, build the noisy `info` dict with `UWMForwardProcess`, run one denoiser forward, backprop through LoRA params only.

In [8]:
forward_process = UWMForwardProcess(
    max_diff_steps=cfg.denoiser.num_noise_levels,
    device=device,
)

trainable_params = [p for p in denoiser.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable_params, lr=1e-4)

N_STEPS = 20  # toy

for step, batch in enumerate(loader):
    if step >= N_STEPS:
        break

    imgs    = interpolate(batch["image"].view(-1, *batch["image"].shape[-3:]), RES)
    imgs    = imgs.view(BATCH, WINDOW, *imgs.shape[-3:]).to(device)
    actions = batch["action"][..., :cfg.denoiser.n_actions].to(device)  # (B, T, n_act)

    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        z_clean = tokenizer.encode(imgs)            # (B, T, N_lat, D_lat)
    a_clean = actions.unsqueeze(-2)                 # (B, T, 1, n_act)

    info = forward_process(z_clean.float(), a_clean.float())

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        obs_loss, act_loss = compute_uwm_loss(info, denoiser, device=device)
        loss = obs_loss + act_loss

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    print(f"step {step:3d}  mode={info['mode']:7s}  obs={obs_loss.item():.4f}  act={act_loss.item():.4f}")

step   0  mode=policy   obs=0.0007  act=0.0048
step   1  mode=wm       obs=0.0006  act=0.0000
step   2  mode=wm       obs=0.0011  act=0.0000
step   3  mode=forcing  obs=0.0020  act=0.0058
step   4  mode=id       obs=0.0000  act=0.0033
step   5  mode=policy   obs=0.0007  act=0.0039
step   6  mode=forcing  obs=0.0012  act=0.0046
step   7  mode=policy   obs=0.0010  act=0.0090
step   8  mode=forcing  obs=0.0019  act=0.0091
step   9  mode=video    obs=0.0017  act=0.0000
step  10  mode=wm       obs=0.0006  act=0.0000
step  11  mode=forcing  obs=0.0026  act=0.0251
step  12  mode=policy   obs=0.0010  act=0.0135
step  13  mode=id       obs=0.0000  act=0.0066
step  14  mode=id       obs=0.0000  act=0.0056
step  15  mode=policy   obs=0.0014  act=0.0067
step  16  mode=forcing  obs=0.0017  act=0.0127
step  17  mode=policy   obs=0.0010  act=0.0050
step  18  mode=wm       obs=0.0011  act=0.0000
step  19  mode=video    obs=0.0013  act=0.0000


## 5. Save & reload the adapter
`save_pretrained` writes only the LoRA matrices (a few MB), not the base weights. To reload, rebuild the base `DenoiserWrapper` exactly the same way and re-wrap it.

In [ ]:
ADAPTER_DIR = "./lora_adapter_toy"
denoiser.save_pretrained(ADAPTER_DIR)
import os
os.listdir(ADAPTER_DIR)

In [ ]:
from peft import PeftModel

base = load_denoiser(cfg, device)                           # frozen pretrained weights
finetuned = PeftModel.from_pretrained(base, ADAPTER_DIR)   # adds LoRA back on top
finetuned.eval();

### Optional: merge LoRA into the base for plain inference
After `merge_and_unload()` you get back a vanilla `DenoiserWrapper` with `W_eff = W + (alpha/r)·B·A` baked into the linear weights. Use this for the existing samplers (`unified_flowmatching_sampler`, etc.) without any PEFT runtime overhead.

In [ ]:
merged = finetuned.merge_and_unload()  # returns the underlying DenoiserWrapper, LoRA folded into W
type(merged), type(merged.model)        # DenoiserWrapper, DreamerV4Denoiser